# Survival Analysis

### Cox Model
- coef
- exp(coef)
- p-value

In [3]:
import pandas as pd
from lifelines import CoxPHFitter

In [4]:
#Load data
DATA_PATH = r"C:\Users\HP\Desktop\BIA\Breast_Cancer_Risk_Prediction\Data\Breast Cancer METABRIC.csv"
df = pd.read_csv(DATA_PATH)
df.head()


,Patient ID,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Cancer Type Detailed,Cellularity,Chemotherapy,Pam50 + Claudin-low subtype,Cohort,ER status measured by IHC,...,Overall Survival Status,PR Status,Radio Therapy,Relapse Free Status (Months),Relapse Free Status,Sex,3-Gene classifier subtype,Tumor Size,Tumor Stage,Patient's Vital Status
0,MB-0000,75.65,Mastectomy,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,No,claudin-low,1.0,Positve,...,Living,Negative,Yes,138.65,Not Recurred,Female,ER-/HER2-,22.0,2.0,Living
1,MB-0002,43.19,Breast Conserving,Breast Cancer,Breast Invasive Ductal Carcinoma,High,No,LumA,1.0,Positve,...,Living,Positive,Yes,83.52,Not Recurred,Female,ER+/HER2- High Prolif,10.0,1.0,Living
2,MB-0005,48.87,Mastectomy,Breast Cancer,Breast Invasive Ductal Carcinoma,High,Yes,LumB,1.0,Positve,...,Deceased,Positive,No,151.28,Recurred,Female,NaN,15.0,2.0,Died of Disease
3,MB-0006,47.68,Mastectomy,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,Yes,LumB,1.0,Positve,...,Living,Positive,Yes,162.76,Not Recurred,Female,NaN,25.0,2.0,Living
4,MB-0008,76.97,Mastectomy,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,Yes,LumB,1.0,Positve,...,Deceased,Positive,Yes,18.55,Recurred,Female,ER+/HER2- High Prolif,40.0,2.0,Died of Disease


In [6]:
# Create event column
df['event'] = (df['Overall Survival Status'] == 'Deceased').astype(int)

# Create time column
df['time'] = df['Overall Survival (Months)']

In [7]:
print(df.columns)

Index(['Patient ID', 'Age at Diagnosis', 'Type of Breast Surgery',
       'Cancer Type', 'Cancer Type Detailed', 'Cellularity', 'Chemotherapy',
       'Pam50 + Claudin-low subtype', 'Cohort', 'ER status measured by IHC',
       'ER Status', 'Neoplasm Histologic Grade',
       'HER2 status measured by SNP6', 'HER2 Status',
       'Tumor Other Histologic Subtype', 'Hormone Therapy',
       'Inferred Menopausal State', 'Integrative Cluster',
       'Primary Tumor Laterality', 'Lymph nodes examined positive',
       'Mutation Count', 'Nottingham prognostic index', 'Oncotree Code',
       'Overall Survival (Months)', 'Overall Survival Status', 'PR Status',
       'Radio Therapy', 'Relapse Free Status (Months)', 'Relapse Free Status',
       'Sex', '3-Gene classifier subtype', 'Tumor Size', 'Tumor Stage',
       'Patient's Vital Status', 'event', 'time'],
      dtype='object')


In [8]:
df_cox = df[['time', 'event', 'Age at Diagnosis', 'Tumor Size']]

In [10]:
df_cox = df_cox.dropna()

In [11]:
# Remove NaN values
df_cox = df_cox.dropna()

In [12]:
# Model fit
cph = CoxPHFitter()
cph.fit(df_cox, duration_col='time', event_col='event')

<lifelines.CoxPHFitter: fitted with 1955 total observations, 827 right-censored observations>

In [13]:
# Summary
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 1955 total observations, 827 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 1955
number of events observed = 1128
   partial log-likelihood = -7606.25
         time fit was run = 2026-04-14 05:12:48 UTC

---
                  coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                         
Age at Diagnosis  0.03      1.03      0.00            0.03            0.04                1.03                1.04
Tumor Size        0.02      1.02      0.00            0.01            0.02                1.01                1.02

                  cmp to     z      p  -log2(p)
covariate                                      
Age at Diagnosis    0.00 13.21 <0.005    130.00
Tumor Size          0.00 10.16 <0.005     78.09
---
Concordance = 0.62
Partial AIC = 15216.50
log-likelihood ratio test = 265.67 on 2 df
-log2(p) of ll-ratio test = 191.64

In [15]:
# Create target variable (10-year mortality)
df['target_10yr'] = ((df['time'] <= 120) & (df['event'] == 1)).astype(int)

In [16]:
df['target_10yr'].value_counts()

target_10yr
0    1749
1     760
Name: count, dtype: int64